<a href="https://colab.research.google.com/github/yWolfy/PRODIGY_ML_01/blob/main/Retrieval_Augmented_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval Augmented Generation Using CLIP

In this notebook, I walk through how to build a multimodal RAG (Retrieval-Augmented Generation) system using LangChain.
The goal is to create an application capable of working with multiple types of data, such as text, images, and documents.

- I show how to set up a multimodal RAG pipeline that can process and combine different data formats.

- I use LangChain to load, manage, and connect multiple data sources (for example PDFs and images) into a single system.

- I demonstrate how to transform and index the data into a vector database to enable semantic search.

Finally, I explain how the system can retrieve relevant information from these sources and generate answers or summaries by combining text and image-based context.

💡 In summary, this notebook is a hands-on tutorial aimed at developers or AI practitioners who want to learn how to build a multimodal RAG application using LangChain, capable of reasoning over both text and images within a Google Colab environment.

In [1]:
!pip install PyMuPDF
!pip install langchain
!pip install langchain_text_splitters
!pip install langchain_community
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 2.3 MB/s eta 0:00:00


In [2]:
# PyMuPDF for PDF processing
import fitz
# LangChain's Document class
from langchain_core.documents import Document
# Hugging Face Transformers for CLIP model and processor
from transformers import CLIPProcessor, CLIPModel
# Python Imaging Library for image manipulation
from PIL import Image
# PyTorch for tensor operations
import torch
# NumPy for numerical operations
import numpy as np
# LangChain chat model initialization (though later switched to Google)
from langchain.chat_models import init_chat_model
# LangChain for prompt templating
from langchain_core.prompts import PromptTemplate
# LangChain for human message objects
from langchain_core.messages import HumanMessage
# Scikit-learn for cosine similarity calculations
from sklearn.metrics.pairwise import cosine_similarity
# Operating system module for environment variables
import os
# For encoding/decoding base64 data
import base64
# For working with I/O streams
import io
# LangChain for splitting text into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
# LangChain for FAISS vector store integration
from langchain_community.vectorstores import FAISS

In [3]:
from google.colab import userdata

In [4]:
###Clip Model
from dotenv import load_dotenv
# Attempt to load environment variables from a .env file, if present.
# This is a common practice for managing sensitive information like API keys.
load_dotenv()

## set up the environment
# Retrieve the GEMINI_API_KEY from environment variables. This variable should ideally
# be stored securely, e.g., in Colab secrets or a .env file.
gemini_api_key = os.getenv("GEMINI_API_KEY")

# Check if the API key was found in the environment variables.
if gemini_api_key is None:
    # If not found, attempt to load it from Google Colab's user data secrets.
    try:
        from google.colab import userdata
        gemini_api_key = userdata.get("GEMINI_API_KEY")
        if gemini_api_key is None:
            # If still not found, raise an error indicating the API key is missing.
            raise ValueError("GEMINI_API_KEY not found in Colab secrets.")
        print("GEMINI_API_KEY loaded from Colab secrets.")
    except ImportError:
        # Handle cases where `google.colab.userdata` is not available (e.g., not running in Colab).
        print("google.colab.userdata not available (not running in Colab).")
        raise ValueError("GEMINI_API_KEY not found in .env or as environment variable.")
    except Exception as e:
        # Catch any other exceptions during Colab secrets loading.
        raise ValueError(f"Error loading GEMINI_API_KEY from Colab secrets: {e}")

# Final check to ensure the API key is set.
if gemini_api_key is None:
    raise ValueError("GEMINI_API_KEY is not set. Please provide your API key via .env file or Colab secrets.")
else:
    # Set the GOOGLE_API_KEY environment variable. Langchain's Google models typically look for this.
    os.environ["GOOGLE_API_KEY"] = gemini_api_key
    print("GEMINI_API_KEY successfully set.")


### initialize the Clip Model for unified embeddings
# Import necessary classes for CLIP model from Hugging Face Transformers library.
from transformers import CLIPProcessor, CLIPModel
# Load the pre-trained CLIP model (Vision Transformer, base patch 32).
# CLIP is crucial for generating unified embeddings for both text and images.
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
# Load the corresponding processor for the CLIP model. The processor handles tokenization and image preprocessing.
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
# Set the model to evaluation mode, which disables dropout and batch normalization updates.
clip_model.eval()

GEMINI_API_KEY loaded from Colab secrets.
GEMINI_API_KEY successfully set.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [5]:
### Embedding functions
def embed_image(image_data):
    """Embed image using CLIP"""
    if isinstance(image_data, str):  # If path
        image = Image.open(image_data).convert("RGB")
    else:  # If PIL Image
        image = image_data

    inputs=clip_processor(images=image,return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        # Normalize embeddings to unit vector
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()

def embed_text(text):
    """Embed text using CLIP."""
    inputs = clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77  # CLIP's max token length
    )
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
        # Normalize embeddings
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()

In [6]:
from google.colab import files

# This will open a file upload dialog. Please upload 'multimodal_sample.pdf'
uploaded = files.upload()

Saving multimodal_sample.pdf to multimodal_sample.pdf


In [7]:
## Process PDF
pdf_path="multimodal_sample.pdf"
doc=fitz.open(pdf_path)
# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {}  # Store actual image data for LLM

# Text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [8]:
doc

Document('multimodal_sample.pdf')

In [10]:
for i,page in enumerate(doc):
    ## Process Text Content
    # Extract text from the current page
    text=page.get_text()
    # Check if the extracted text is not empty
    if text.strip():
        ## Create a temporary Document object for text splitting
        # This ensures metadata (like page number and type) is preserved
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        # Split the text into smaller, manageable chunks to improve retrieval accuracy
        text_chunks = splitter.split_documents([temp_doc])

        # Embed each text chunk using the CLIP model
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk) # Store the document with its metadata



    ## Process Images Content
    ## For each image, three important actions are performed:
    ## 1. Convert the image extracted from PDF to PIL format for consistent handling.
    ## 2. Store the image as base64 for compatibility with multimodal LLMs like GPT-4V/Gemini-Pro-Vision.
    ## 3. Create a CLIP embedding for the image, enabling its retrieval based on visual similarity.

    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            # Get the cross-reference (xref) for the image
            xref = img[0]
            # Extract image data from the PDF using its xref
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            # Convert the raw image bytes to a PIL Image object, ensuring RGB format
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

            # Create a unique identifier for each image for easy reference
            image_id = f"page_{i}_img_{img_index}"

            # Encode the PIL image to base64 string, which is required by multimodal LLMs
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG") # Save as PNG to buffer
            img_base64 = base64.b64encode(buffered.getvalue()).decode() # Get base64 string
            image_data_store[image_id] = img_base64 # Store base64 image data

            # Embed the PIL image using the CLIP model to get its vector representation
            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)

            # Create a Document object for the image, referencing its ID and metadata
            image_doc = Document(
                page_content=f"[Image: {image_id}]", # Placeholder text for the image
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc) # Store the image document

        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue # Continue to the next image even if one fails

doc.close() # Close the PDF document after processing all pages

In [11]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
 Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')]

In [12]:
# Create unified FAISS vector store with CLIP embeddings
embeddings_array = np.array(all_embeddings)
embeddings_array

array([[-0.00267244,  0.01282997, -0.05183141, ..., -0.00385083,
         0.0297772 , -0.00010686],
       [ 0.01732338, -0.01327688, -0.02427028, ...,  0.08994054,
        -0.00272158,  0.03253039]], dtype=float32)

In [13]:
(all_docs,embeddings_array)

([Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
  Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')],
 array([[-0.00267244,  0.01282997, -0.05183141, ..., -0.00385083,
          0.0297772 , -0.00010686],
        [ 0.01732338, -0.01327688, -0.02427028, ...,  0.08994054,
         -0.00272158,  0.03253039]], dtype=float32))

In [14]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.8 MB/s eta 0:00:00


In [15]:
# Create custom FAISS index since we have precomputed embeddings
# FAISS (Facebook AI Similarity Search) is a library for efficient similarity search
# and clustering of dense vectors. Here, we're using it to store our CLIP embeddings.
vector_store = FAISS.from_embeddings(
    # The 'text_embeddings' parameter expects a list of tuples, where each tuple contains
    # the document content (text or a placeholder for an image) and its corresponding embedding.
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding=None,  # We're using precomputed embeddings, so no embedding function is needed here.
    # 'metadatas' stores additional information about each document, such as page number and type (text/image).
    metadatas=[doc.metadata for doc in all_docs]
)
# Display the created vector store object (for verification)
vector_store

In [16]:
!pip install langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.9/234.9 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.43.0
    Uninstalling google-auth-2.43.0:
      Successfully uninstalled google-auth-2.43.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.55.0
    Uninstalling google-genai-1.55.0:
      Successfully uninstalled google-genai-1.55.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, bu

In [17]:
# Initialize Gemini-Pro-Vision model
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)
llm

ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', temperature=0.1, client=<google.genai.client.Client object at 0x782fc96a54f0>, default_metadata=(), model_kwargs={})

In [18]:
def retrieve_multimodal(query, k=5):
    """Unified retrieval using CLIP embeddings for both text and images."""
    # Embed query using CLIP: The user's natural language query is converted into a numerical vector
    # using the CLIP text encoder. This embedding allows the system to compare the query semantically
    # with the stored document embeddings.
    query_embedding = embed_text(query)

    # Search in unified vector store: The generated query embedding is used to perform a similarity search
    # against the FAISS vector store, which contains embeddings of both text chunks and images.
    # 'k' determines the number of top most similar documents (text or image) to retrieve.
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )

    return results

In [19]:
def create_multimodal_message(query, retrieved_docs):
    """Create a message with both text and images for GPT-4V/Gemini-Pro-Vision."""
    content = []

    # Add the original query to the message content, formatted as a question.
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n"
    })

    # Separate retrieved documents into text and image categories based on their metadata.
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]

    # If text documents are retrieved, format them as text excerpts with their page numbers
    # and add them to the message content. This provides textual context to the LLM.
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })

    # For each retrieved image document, retrieve its base64 encoded data from the `image_data_store`.
    # Then, add both a textual reference to the image (with its page number) and the base64 image data
    # to the message content. This allows multimodal LLMs to 'see' and reason about the images.
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]\n"
            })
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })

    # Add a final instruction to the LLM, guiding it to answer based on the provided context (text and images).
    content.append({
        "type": "text",
        "text": "\n\nPlease answer the question based on the provided text and images."
    })

    # Return a HumanMessage object, which is LangChain's representation of a message from a human user,
    # containing all the multimodal content.
    return HumanMessage(content=content)

In [20]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal RAG."""
    # Step 1: Retrieve relevant documents (text and/or images) from the vector store.
    # The 'retrieve_multimodal' function uses the CLIP model to find documents semantically
    # similar to the user's query. 'k=5' means it will retrieve the top 5 most relevant items.
    context_docs = retrieve_multimodal(query, k=5)

    # Step 2: Create a multimodal message for the LLM.
    # This function formats the user's query and the retrieved documents (including base64 encoded images)
    # into a structure that can be understood by a multimodal LLM (like Gemini-Pro-Vision).
    message = create_multimodal_message(query, context_docs)

    # Step 3: Invoke the LLM with the multimodal message.
    # The LLM processes the query along with the provided textual and visual context
    # to generate an informed response.
    response = llm.invoke([message])

    # Print retrieved context info (for debugging and transparency).
    # This section helps visualize which documents were considered by the RAG pipeline.
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")

    # Return the content of the LLM's response.
    return response.content

In [21]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "What does the chart on page 1 show about revenue trends?",
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]

    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)


Query: What does the chart on page 1 show about revenue trends?
--------------------------------------------------

Retrieved 2 documents:
  - Text from page 0: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illust...
  - Image from page 0


Answer: The chart on page 0 (which the question likely refers to as "page 1") shows a clear upward trend in revenue across Q1, Q2, and Q3.

Specifically:
*   There are three bars of increasing height, representing the revenue for each quarter.
*   The first bar (blue) is the shortest, indicating Q1 had the lowest revenue.
*   The second bar (green) is taller than the first, showing that Q2 revenue increased compared to Q1.
*   The third bar (red) is the tallest, illustrating that Q3 had the highest revenue, confirming the "highest growth recorded in Q3" as stated in the text.

In summary, the chart visually confirms that revenue grew steadily from Q1 to Q2, and then experienced its highest growth in Q

In [22]:
import google.generativeai as genai
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Available Gemini Models that support generateContent:")
for model in genai.list_models():
  if "generateContent" in model.supported_generation_methods:
    print(model.name)

print("\nPlease ensure the model you are trying to use is in this list.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available Gemini Models that support generateContent:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-c